In [9]:
import numpy as np
import pandas as pd
from scipy.stats import kurtosis, skew

# =================================================================
# SECTION 1: SECURE AUTOMATIC GENERATION OF DATASETS
# =================================================================
# Setting a seed ensures the generated "mock data" is identical every run
np.random.seed(42)

# 1. Automatically generate the UCI Early Stage Dataset structure
uci_mock = pd.DataFrame({
    "Age": np.random.randint(20, 70, size=50),
    "Gender": np.random.choice(["Male", "Female"], size=50),
    "Polyuria": np.random.choice(["Yes", "No"], size=50),
    "Class": np.random.choice(["Positive", "Negative"], size=50)
})
uci_mock.to_csv("uci_diabetes.csv", index=False)

# 2. Automatically generate the Pima Indians Dataset structure
pima_mock = pd.DataFrame({
    "Pregnancies": np.random.randint(0, 15, size=50),
    "Glucose": np.random.randint(70, 200, size=50),
    "BloodPressure": np.random.randint(40, 122, size=50),
    "SkinThickness": np.random.randint(0, 50, size=50),  # Contains 0s
    "Insulin": np.random.randint(0, 300, size=50),        # Contains 0s
    "BMI": np.random.uniform(18.0, 45.0, size=50),
    "DiabetesPedigreeFunction": np.random.uniform(0.1, 2.5, size=50),
    "Age": np.random.randint(21, 81, size=50),
    "Outcome": np.random.choice([0, 1], size=50)
})

# Inject manual 0s into clinical fields to simulate real-world data issues
pima_mock.loc[np.random.choice(50, 5, replace=False), "BloodPressure"] = 0
pima_mock.loc[np.random.choice(50, 8, replace=False), "SkinThickness"] = 0
pima_mock.loc[np.random.choice(50, 10, replace=False), "Insulin"] = 0

pima_mock.to_csv("pima_diabetes.csv", index=False)


# =================================================================
# SECTION 2: UNIVARIATE STATISTICAL ANALYSIS ENGINE
# =================================================================
def perform_univariate_analysis(df, dataset_name, target_columns):
    results = {}

    for col in target_columns:
        if col in df.columns:
            clean_series = df[col].dropna()

            # Descriptive statistics
            frequency = clean_series.count()
            mean_val = clean_series.mean()
            median_val = clean_series.median()

            mode_series = clean_series.mode()
            mode_val = mode_series.iloc[0] if not mode_series.empty else np.nan

            variance_val = clean_series.var(ddof=1)
            std_dev_val = clean_series.std(ddof=1)

            # Safeguard calculation for Skewness and Kurtosis
            if std_dev_val > 0 and len(clean_series) > 2:
                skewness_val = skew(clean_series, bias=False)
                kurtosis_val = kurtosis(clean_series, bias=False)
            else:
                skewness_val, kurtosis_val = np.nan, np.nan

            results[col] = {
                "Count (Freq)": int(frequency),
                "Mean": round(mean_val, 4),
                "Median": round(median_val, 4),
                "Mode": round(mode_val, 4),
                "Variance": round(variance_val, 4),
                "Std Deviation": round(std_dev_val, 4),
                "Skewness": round(skewness_val, 4) if pd.notna(skewness_val) else np.nan,
                "Kurtosis": round(kurtosis_val, 4) if pd.notna(kurtosis_val) else np.nan,
            }

    analysis_table = pd.DataFrame(results).T
    print(f"\n================ {dataset_name} ================")
    print(analysis_table.to_string())
    return analysis_table


# =================================================================
# SECTION 3: CORE RUN PIPELINE
# =================================================================
# Load the newly created files securely from your environment
uci_df = pd.read_csv("uci_diabetes.csv")
pima_df = pd.read_csv("pima_diabetes.csv")

# Clean zero placeholders in biological clinical indicators
zero_sensitive_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for col in zero_sensitive_cols:
    pima_df[col] = pima_df[col].replace(0, np.nan)

# Map continuous parameters
uci_cols = ["Age"]
pima_cols = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
]

# Run analysis 
uci_summary = perform_univariate_analysis(uci_df, "UCI Early Stage Diabetes Dataset", uci_cols)
pima_summary = perform_univariate_analysis(pima_df, "Pima Indians Diabetes Dataset (Cleaned Zeros)", pima_cols)




================ UCI Early Stage Diabetes Dataset ================
     Count (Freq)   Mean  Median  Mode  Variance  Std Deviation  Skewness  Kurtosis
Age          50.0  43.68    43.0  40.0  192.9567        13.8909    0.0929   -1.0407

================ Pima Indians Diabetes Dataset (Cleaned Zeros) ================
                          Count (Freq)      Mean    Median      Mode   Variance  Std Deviation  Skewness  Kurtosis
Pregnancies                       50.0    7.6200    8.0000    8.0000    16.5261         4.0652   -0.3851   -0.8603
Glucose                           50.0  147.4000  157.5000  182.0000  1664.8571        40.8027   -0.4151   -1.0207
BloodPressure                     45.0   77.1778   72.0000   72.0000   505.4677        22.4826    0.4865   -0.8390
SkinThickness                     42.0   26.7381   28.0000   32.0000   203.5151        14.2659   -0.3625   -0.7438
Insulin                           40.0  136.2750  141.5000   91.0000  6873.9481        82.9093    0.3013   -